# Feature Table 생성

## 개요
이 노트북은 ML 학습에 사용할 피처 테이블을 생성합니다.  
**다음 ML 노트북은 이 노트북이 생성한 CSV를 기준으로 실행됩니다.**

---

## 버전 관리 규칙

| 버전 | 셀 범위 | 출력 파일 |
|------|---------|-----------|
| v1 | 셀 1~8 | `data/processed/feature_table_v1.csv` |
| v2 | + 셀 9~ | `data/processed/feature_table_v2.csv` |

> ⚠️ **v1은 수정 금지** — 모든 모델의 성능 기준점(베이스라인)  
> 새 피처 추가 시 셀을 아래에 **추가**하고 버전만 올릴 것  
> 기존 컬럼명 변경 금지 — 새 피처 추가만 허용  
> 노트북 동시 편집 금지 — 커밋/푸시 후 다음 사람 pull  

---

## 타겟 변수 3개 (고정)

| 컬럼 | 설명 | 용도 |
|------|------|------|
| `total_audience` | `MAX(audiAcc)` — 최종 누적 관객수 | 회귀 타겟 원본 |
| `log_audience` | `log1p(total_audience)` | 회귀 타겟 (실제 학습용) |
| `hit_class` | 관객수 구간 0~3 | 분류 타겟 |

```
hit_class 기준 (셀 0의 HIT_BINS에서 수정 가능):
  0: ~10만      (소규모)
  1: 10만~100만  (중규모)
  2: 100만~500만 (흥행)
  3: 500만~      (대흥행)
```

---

## ⚠️ 데이터 누수 원칙
**개봉 전에 알 수 있는 정보만 피처로 사용합니다.**

| ✅ 사용 가능 | ❌ 사용 금지 |
|-------------|-------------|
| genre, rating, nation, runtime, open_date | rank, audiCnt, salesAmt, scrnCnt (개봉 후 데이터) |
| 감독/배우/배급사/제작사의 **과거작** 실적 | 해당 영화 자신의 box office 데이터 |
| 개봉일 기준 시즌/공휴일 정보 | — |

---
## ⚙️ 셀 0. 환경 설정

- `VERSION` : 생성할 CSV 버전 — 새 피처 추가 시 `v2`, `v3`으로 올릴 것
- `HIT_BINS` : `hit_class` 구간 기준값. EDA 결과 보고 조정
- `STAR_POWER_TOP_N` : 주연 배우 Star Power 계산에 사용할 배우 수

In [12]:
import os
import warnings
import numpy as np
import pandas as pd

from data.db import db

warnings.filterwarnings('ignore')

# ── 설정값 (필요 시 조정) ──────────────────────────────
VERSION          = 'v1'
OUTPUT_DIR       = 'data/processed'
STAR_POWER_TOP_N = 3   # 출연진 중 Star Power 계산에 포함할 배우 수

# hit_class 구간 기준 (단위: 명)
# 0: ~10만 / 1: 10만~100만 / 2: 100만~500만 / 3: 500만~
HIT_BINS   = [0, 100_000, 1_000_000, 5_000_000, float('inf')]
HIT_LABELS = [0, 1, 2, 3]
# ───────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = f'{OUTPUT_DIR}/feature_table_{VERSION}.csv'

print(f'✅ 설정 완료')
print(f'   버전      : {VERSION}')
print(f'   출력 경로 : {OUTPUT_PATH}')
print(f'   hit_class : {HIT_BINS}')

✅ 설정 완료
   버전      : v1
   출력 경로 : data/processed/feature_table_v1.csv
   hit_class : [0, 100000, 1000000, 5000000, inf]


---
## ⚙️ 셀 0.5 sql 쿼리문 정리

- boxoffice_rows
- movie_rows
- holiday_rows
- casting_rows
- company_rows
- new_rows

In [13]:
# 1. 타겟 변수 생성 위한 boxoffice_rows
boxoffice_rows = db.fetch_all("""
    SELECT
        movie_id,
        MAX(audiAcc)  AS total_audience,
        MAX(salesAcc) AS total_sales
    FROM daily_box_office
    GROUP BY movie_id
""")
# 2. 영화 메타 피처 생성 위한 movie_rows
movie_rows = db.fetch_all("""
    SELECT movie_id, title, genre, rating, nation, open_date, runtime
    FROM movies
""")

# 3. 기간/계절 피처 생성 위한 공휴일 holidays_rows
holiday_rows = db.fetch_all('SELECT holiday_date, is_weekend_effect FROM holidays')

# 4. 영화-영화인-역할 전체 로드 casting_rows
casting_rows = db.fetch_all("""
    SELECT
        mc.movie_id,
        mc.person_id,
        mc.cast_role,
        p.name
    FROM movie_casting mc
    JOIN people p ON mc.person_id = p.person_id
""")
# 5. 영화-영화사 관계 로드 company_rows 
company_rows = db.fetch_all("""
    SELECT cp.movie_id, cp.company_id, cp.part_role, c.company_name
    FROM company_part cp
    JOIN companys c ON cp.company_id = c.company_id
    WHERE cp.part_role IN ('배급사', '제작사')
""")
# 6. 경쟁 환경 피처 위한 신작 개봉일 목록 (rankOldAndNew = 'NEW' 인 첫 등장일) new_rows
new_rows = db.fetch_all("""
    SELECT movie_id, MIN(target_date) AS first_date
    FROM daily_box_office
    WHERE rankOldAndNew = 'NEW'
    GROUP BY movie_id
""")
# 7. 개봉 직전 7일 시장 평균 관객수 ── market_rows
market_rows = db.fetch_all("""
    SELECT target_date, total_audi
    FROM daily_market_stats
    ORDER BY target_date
""")

---
## 📥 셀 1. 타겟 변수 생성 (from `daily_box_office`)

```
MAX(audiAcc) 를 사용하는 이유:
  audiAcc = 누적값 → 날짜가 지날수록 단조 증가
  SUM(audiAcc) → 누적값을 또 더하므로 완전히 틀린 값 ❌
  MAX(audiAcc) → 마지막 날의 누적값 = 최종 총 관객수  ✅
```

In [14]:
# boxoffice_rows = db.fetch_all("""
#     SELECT
#         movie_id,
#         MAX(audiAcc)  AS total_audience,
#         MAX(salesAcc) AS total_sales
#     FROM daily_box_office
#     GROUP BY movie_id
# """)
df = pd.DataFrame(boxoffice_rows)
df['total_audience'] = pd.to_numeric(df['total_audience'], errors='coerce')
df['total_sales']    = pd.to_numeric(df['total_sales'],    errors='coerce')

# 타겟 변수 생성
df['log_audience'] = np.log1p(df['total_audience'])
df['hit_class']    = pd.cut(
    df['total_audience'],
    bins=HIT_BINS,
    labels=HIT_LABELS,
    right=False
).astype('Int64')  # nullable int (NaN 허용)

print(f'✅ 타겟 변수 생성: {len(df)}편')
print(f'\nhit_class 분포:')
label_map = {0: '0 (~10만)', 1: '1 (10만~100만)', 2: '2 (100만~500만)', 3: '3 (500만~)'}
print(df['hit_class'].value_counts().sort_index().rename(label_map).to_string())
print(f'\ntotal_audience 기술통계:')
print(df['total_audience'].describe().apply(lambda x: f'{x:,.0f}'))

✅ 타겟 변수 생성: 3943편

hit_class 분포:
hit_class
0 (~10만)         2129
1 (10만~100만)     1181
2 (100만~500만)     523
3 (500만~)         110

total_audience 기술통계:
count         3,943
mean        656,856
std       1,595,571
min             259
25%          16,994
50%          77,531
75%         480,092
max      17,583,608
Name: total_audience, dtype: str


---
## 🎬 셀 2. 영화 메타 피처 (from `movies`)

개봉 전에 확정되는 정보이므로 모두 사용 가능합니다.

| 피처 | 원본 컬럼 | 변환 방법 |
|------|----------|-----------|
| `genre` | `genre` | 문자열 카테고리 |
| `rating_encoded` | `rating` | 순서형 인코딩 (전체=0 < 12세=1 < 15세=2 < 청불=3) |
| `is_korean` | `nation` | 이진 (한국=1, 외국=0) |
| `runtime` | `runtime` | 수치형 그대로 (분 단위) |

In [15]:
# movie_rows = db.fetch_all("""
#     SELECT movie_id, title, genre, rating, nation, open_date, runtime
#     FROM movies
# """)
df_movies = pd.DataFrame(movie_rows)
df_movies['open_date'] = pd.to_datetime(df_movies['open_date'], errors='coerce')
df_movies['runtime']   = pd.to_numeric(df_movies['runtime'], errors='coerce')

# ── 결측치 / 센티널 값 처리 ──
# '미정'은 수집 단계의 기본값 → 실제 결측으로 변환
df_movies['genre']  = df_movies['genre'].replace('미정', '기타').fillna('기타')
df_movies['rating'] = df_movies['rating'].replace('미정', np.nan)
df_movies['nation'] = df_movies['nation'].replace('미정', np.nan)
# open_date 1900-01-01 = 개봉일 미상 → NaT 처리
df_movies.loc[df_movies['open_date'].dt.year == 1900, 'open_date'] = pd.NaT
# runtime 0 = 미상 → NaN 처리
df_movies.loc[df_movies['runtime'] == 0, 'runtime'] = np.nan
df_movies['runtime'] = df_movies['runtime'].fillna(df_movies['runtime'].median())

# One-Hot 대신 Label Encoding , 아래 코드 없으면 문자열 카테고리로 저장
# genre_categories = df_movies['genre'].astype('category')
# df_movies['genre'] = genre_categories.cat.codes  # 0부터 자동 부여

# ── rating 순서형 인코딩 ──
RATING_ORDER = {
    # 0: 전체관람가 수준
    "전체관람가": 0,
    "연소자관람가": 0,
    "모든 관람객이 관람할 수 있는 등급": 0,
    "전체": 0,
    
    # 1: 12세 관람가 수준
    "12세이상관람가": 1,
    "중학생이상관람가": 1,
    "12세관람가": 1,
    "12세": 1,
    "미정": 1,
    
    # 2: 15세 관람가 수준
    "15세이상관람가": 2,
    "고등학생이상관람가": 2,
    "15세관람가": 2,
    "15세 미만인 자는 관람할 수 없는 등급": 2,
    "15세": 2,
    
    # 3: 청소년 관람불가 수준
    "청소년관람불가": 3,
    "연소자관람불가": 3,
    "18세 미만인 자는 관람할 수 없는 등급": 3,
    "18세관람가": 3,
    "미성년자관람불가": 3
}

df_movies['rating_encoded'] = df_movies['rating'].map(RATING_ORDER)
# 매핑 안 된 값 → 중간값(2)으로 대체(고등학생이상 관람가 등 매핑 안 된 값 2로 채움)
df_movies['rating_encoded'] = df_movies['rating_encoded'].fillna(2).astype(int)

# ── 국가: 한국 여부 이진화 ──
df_movies['is_korean'] = (df_movies['nation'] == '한국').astype(int)

# ── 기본 테이블에 병합 ──
# print("genre 매핑:", dict(enumerate(genre_categories.cat.categories)))

meta_cols = ['movie_id', 'title', 'open_date', 'runtime', 'rating_encoded', 'is_korean', 'genre']
df = df.merge(df_movies[meta_cols], on='movie_id', how='left')

print(f'✅ 영화 메타 피처 병합 완료')
print(f'   genre 종류     : {df_movies["genre"].nunique()}개 → genre 1개 컬럼')
print(f'   rating 분포:\n{df_movies["rating"].value_counts().to_string()}')

✅ 영화 메타 피처 병합 완료
   genre 종류     : 21개 → genre 1개 컬럼
   rating 분포:
rating
15세이상관람가                   1289
12세이상관람가                   1026
전체관람가                      1023
청소년관람불가                     511
15세관람가                       14
12세관람가                       13
고등학생이상관람가                     9
연소자관람불가                       7
18세관람가                        7
중학생이상관람가                      6
연소자관람가                        6
15세 미만인 자는 관람할 수 없는 등급        4
18세 미만인 자는 관람할 수 없는 등급        2
                              1
모든 관람객이 관람할 수 있는 등급           1


---
## 📅 셀 3. 시간/계절 피처 (from `movies.open_date` + `holidays`)

| 피처 | 산출 방법 | 의미 |
|------|-----------|------|
| `open_month` | 개봉일에서 추출 | 여름(7~8)/겨울(12~1) 성수기 효과 |
| `open_day_of_week` | 개봉일 요일 (0=월~6=일) | 수요일 개봉 vs 목요일 개봉 |
| `is_holiday_release` | holidays 테이블 JOIN | 공휴일 당일 개봉 여부 |
| `holiday_nearby_count` | 개봉일 ±7일 내 공휴일 수 | 개봉 첫 주 관객 동원 잠재력 |
| `is_summer` | 7~8월 여부 | 성수기 이진 변수 |
| `is_winter` | 12~1월 여부 | 성수기 이진 변수 |

In [16]:
# ── 날짜 기반 피처 ──
df['open_month']       = df['open_date'].dt.month
df['open_day_of_week'] = df['open_date'].dt.dayofweek  # 0=월요일, 6=일요일
df['is_summer']        = df['open_month'].isin([7, 8]).astype(int)
df['is_winter']        = df['open_month'].isin([12, 1]).astype(int)

# ── 공휴일 피처 ──
# holiday_rows = db.fetch_all('SELECT holiday_date, is_weekend_effect FROM holidays')

if holiday_rows:
    df_holidays = pd.DataFrame(holiday_rows)
    df_holidays['holiday_date'] = pd.to_datetime(df_holidays['holiday_date'])
    holiday_np = df_holidays['holiday_date'].values.astype('datetime64[D]')

    def get_holiday_features(open_dt, holiday_np, window=7):
        """
        개봉일 기준 ±window일 이내 공휴일 수와 공휴일 당일 개봉 여부를 반환합니다.
        holidays 테이블의 is_weekend_effect 컬럼은 별도로 JOIN합니다.
        """
        if pd.isna(open_dt):
            return 0, 0
        open_np   = np.datetime64(open_dt, 'D')
        diff_days = np.abs((holiday_np - open_np).astype(int))
        nearby    = int((diff_days <= window).sum())
        is_hol    = int((diff_days == 0).any())
        return is_hol, nearby

    results = df['open_date'].apply(
        lambda d: pd.Series(get_holiday_features(d, holiday_np),
                            index=['is_holiday_release', 'holiday_nearby_count'])
    )
    df = pd.concat([df, results], axis=1)
    print(f'✅ 공휴일 피처 생성 완료')
    print(f'   공휴일 당일 개봉: {df["is_holiday_release"].sum()}편')
    print(f'   ±7일 내 공휴일 평균: {df["holiday_nearby_count"].mean():.2f}일')
else:
    df['is_holiday_release']   = 0
    df['holiday_nearby_count'] = 0
    print('⚠️  holidays 테이블이 비어있어 공휴일 피처를 0으로 설정')

print(f'\n✅ 시간/계절 피처 생성 완료')
print(f'   여름 개봉: {df["is_summer"].sum()}편 / 겨울 개봉: {df["is_winter"].sum()}편')

✅ 공휴일 피처 생성 완료
   공휴일 당일 개봉: 188편
   ±7일 내 공휴일 평균: 0.50일

✅ 시간/계절 피처 생성 완료
   여름 개봉: 613편 / 겨울 개봉: 614편


---
## 🌟 셀 4. Star Power 피처 (from `people` + `movie_casting` + `daily_box_office`)

**핵심 원칙: 해당 영화 개봉일 이전의 과거 실적만 참조**

| 피처 | 산출 방법 | 의미 |
|------|-----------|------|
| `director_avg_audi` | 감독의 과거 영화 평균 관객수 | 감독 흥행력 |
| `director_movie_count` | 감독의 과거 영화 편수 | 감독 경력 |
| `lead_actor_avg_audi` | 주연 배우의 과거 평균 관객수 | 배우 흥행력 |
| `lead_actor_movie_count` | 주연 배우의 과거 영화 편수 | 배우 경력 |
| `cast_max_star_power` | 출연진 중 최고 평균 관객수 | 캐스팅 파워 상한 |

> 신인 감독/배우(과거 데이터 없음)의 경우 → 0으로 설정

In [17]:
# ── 4-0. 영화별 개봉일 맵 준비 ──
# Star Power 계산 시 '개봉일 이전' 필터링에 사용
open_date_map = df.set_index('movie_id')['open_date'].to_dict()

# ── 4-1. 영화-영화인-역할 전체 로드 ──
# casting_rows = db.fetch_all("""
#     SELECT
#         mc.movie_id,
#         mc.person_id,
#         mc.cast_role,
#         p.name
#     FROM movie_casting mc
#     JOIN people p ON mc.person_id = p.person_id
# """)
df_casting = pd.DataFrame(casting_rows)

# ── 4-2. 영화별 최종 관객수 맵 준비 ──
audi_map = df.set_index('movie_id')['total_audience'].to_dict()

def calc_star_power(person_id, target_movie_id, df_casting, audi_map, open_date_map):
    """
    특정 영화인의 Star Power를 계산합니다.
    target_movie_id의 개봉일 이전에 개봉한 과거작의 평균 관객수를 반환합니다.

    Args:
        person_id       : 영화인 고유 ID
        target_movie_id : 기준 영화 ID (이 영화의 개봉일 이전 실적만 참조)

    Returns:
        (avg_audi, movie_count): 평균 관객수, 과거작 편수
    """
    target_open = open_date_map.get(target_movie_id)
    if pd.isna(target_open):
        return 0.0, 0

    # 해당 영화인의 모든 출연작
    person_movies = df_casting[df_casting['person_id'] == person_id]['movie_id'].tolist()

    past_audis = []
    for mid in person_movies:
        if mid == target_movie_id:
            continue  # 자기 자신 제외
        open_dt = open_date_map.get(mid)
        if pd.isna(open_dt) or open_dt >= target_open:
            continue  # 개봉일 이후 작품 제외
        audi = audi_map.get(mid)
        if audi is not None and not np.isnan(float(audi)):
            past_audis.append(float(audi))

    if not past_audis:
        return 0.0, 0
    return float(np.mean(past_audis)), len(past_audis)


# ── 4-3. 감독 Star Power 계산 ──
df_directors = df_casting[df_casting['cast_role'] == '감독'].copy()
# 영화당 감독 1명만 사용
df_directors = df_directors.drop_duplicates(subset='movie_id', keep='first')

director_stats = []
for _, row in df_directors.iterrows():
    avg_a, cnt = calc_star_power(row['person_id'], row['movie_id'], df_casting, audi_map, open_date_map)
    director_stats.append({'movie_id': row['movie_id'], 'director_avg_audi': avg_a, 'director_movie_count': cnt})

df_director_sp = pd.DataFrame(director_stats)
df = df.merge(df_director_sp, on='movie_id', how='left')
df['director_avg_audi']    = df['director_avg_audi'].fillna(0)
df['director_movie_count'] = df['director_movie_count'].fillna(0).astype(int)
print(f'✅ 감독 Star Power 계산 완료')

# ── 4-4. 주연 배우 Star Power 계산 ──
df_actors = df_casting[df_casting['cast_role'] == '배우'].copy()

actor_stats = []
for movie_id, group in df_actors.groupby('movie_id'):
    person_ids = group['person_id'].tolist()

    sp_list = []
    for pid in person_ids:
        avg_a, cnt = calc_star_power(pid, movie_id, df_casting, audi_map, open_date_map)
        sp_list.append((avg_a, cnt))

    # 상위 STAR_POWER_TOP_N명의 평균을 lead_actor_avg_audi로 사용
    sp_list_sorted = sorted(sp_list, key=lambda x: x[0], reverse=True)
    top_n     = sp_list_sorted[:STAR_POWER_TOP_N]
    top_audis = [s[0] for s in top_n if s[0] > 0]

    lead_avg = float(np.mean(top_audis)) if top_audis else 0.0
    lead_cnt = int(np.mean([s[1] for s in top_n])) if top_n else 0
    cast_max = float(sp_list_sorted[0][0]) if sp_list_sorted else 0.0

    actor_stats.append({
        'movie_id':              movie_id,
        'lead_actor_avg_audi':   lead_avg,
        'lead_actor_movie_count': lead_cnt,
        'cast_max_star_power':   cast_max,
    })

df_actor_sp = pd.DataFrame(actor_stats)
df = df.merge(df_actor_sp, on='movie_id', how='left')
df['lead_actor_avg_audi']    = df['lead_actor_avg_audi'].fillna(0)
df['lead_actor_movie_count'] = df['lead_actor_movie_count'].fillna(0).astype(int)
df['cast_max_star_power']    = df['cast_max_star_power'].fillna(0)
print(f'✅ 배우 Star Power 계산 완료')

✅ 감독 Star Power 계산 완료
✅ 배우 Star Power 계산 완료


---
## 🏢 셀 5. Brand Power 피처 (from `companys` + `company_part` + `daily_box_office`)

| 피처 | 산출 방법 | 의미 |
|------|-----------|------|
| `distributor_avg_audi` | 배급사의 과거 영화 평균 관객수 | 배급 역량 |
| `distributor_movie_count` | 배급사의 과거 영화 편수 | 배급 규모 |
| `producer_avg_audi` | 제작사의 과거 영화 평균 관객수 | 제작 역량 |
| `producer_movie_count` | 제작사의 과거 영화 편수 | 제작 규모 |

> Star Power와 동일하게 **개봉일 이전 과거작만** 참조합니다.

In [18]:
# ── 5-1. 영화-영화사 관계 로드 ──
# company_rows = db.fetch_all("""
#     SELECT cp.movie_id, cp.company_id, cp.part_role, c.company_name
#     FROM company_part cp
#     JOIN companys c ON cp.company_id = c.company_id
#     WHERE cp.part_role IN ('배급사', '제작사')
# """)
df_company = pd.DataFrame(company_rows)

def calc_company_power(company_id, target_movie_id, df_company, audi_map, open_date_map):
    """
    특정 영화사의 Brand Power를 계산합니다.
    target_movie_id의 개봉일 이전에 개봉한 해당 영화사 참여작의 평균 관객수를 반환합니다.
    """
    target_open = open_date_map.get(target_movie_id)
    if pd.isna(target_open):
        return 0.0, 0

    company_movies = df_company[df_company['company_id'] == company_id]['movie_id'].tolist()

    past_audis = []
    for mid in company_movies:
        if mid == target_movie_id:
            continue
        open_dt = open_date_map.get(mid)
        if pd.isna(open_dt) or open_dt >= target_open:
            continue
        audi = audi_map.get(mid)
        if audi is not None and not np.isnan(float(audi)):
            past_audis.append(float(audi))

    if not past_audis:
        return 0.0, 0
    return float(np.mean(past_audis)), len(past_audis)


# ── 5-2. 배급사 Brand Power 계산 ──
df_dist = df_company[df_company['part_role'] == '배급사'].drop_duplicates(subset='movie_id', keep='first')

dist_stats = []
for _, row in df_dist.iterrows():
    avg_a, cnt = calc_company_power(row['company_id'], row['movie_id'], df_company, audi_map, open_date_map)
    dist_stats.append({'movie_id': row['movie_id'], 'distributor_avg_audi': avg_a, 'distributor_movie_count': cnt})

df_dist_bp = pd.DataFrame(dist_stats)
df = df.merge(df_dist_bp, on='movie_id', how='left')
df['distributor_avg_audi']    = df['distributor_avg_audi'].fillna(0)
df['distributor_movie_count'] = df['distributor_movie_count'].fillna(0).astype(int)
print(f'✅ 배급사 Brand Power 계산 완료')

# ── 5-3. 제작사 Brand Power 계산 ──
df_prod = df_company[df_company['part_role'] == '제작사'].drop_duplicates(subset='movie_id', keep='first')

prod_stats = []
for _, row in df_prod.iterrows():
    avg_a, cnt = calc_company_power(row['company_id'], row['movie_id'], df_company, audi_map, open_date_map)
    prod_stats.append({'movie_id': row['movie_id'], 'producer_avg_audi': avg_a, 'producer_movie_count': cnt})

df_prod_bp = pd.DataFrame(prod_stats)
df = df.merge(df_prod_bp, on='movie_id', how='left')
df['producer_avg_audi']    = df['producer_avg_audi'].fillna(0)
df['producer_movie_count'] = df['producer_movie_count'].fillna(0).astype(int)
print(f'✅ 제작사 Brand Power 계산 완료')

✅ 배급사 Brand Power 계산 완료
✅ 제작사 Brand Power 계산 완료


---
## ⚔️ 셀 6. 경쟁 환경 피처 (from `daily_box_office` + `daily_market_stats`)

| 피처 | 산출 방법 | 의미 |
|------|-----------|------|
| `same_week_releases` | 개봉일 ±3일 내 다른 신작 수 | 경쟁 강도 |
| `market_avg_audi_7d` | 개봉 직전 7일 시장 평균 관객수 | 시장 활성도 |

> `same_week_releases`는 `daily_box_office`의 `rankOldAndNew = 'NEW'` 조건으로 신작 여부를 판단합니다.

In [19]:
# ── 6-1. 신작 개봉일 목록 (rankOldAndNew = 'NEW' 인 첫 등장일) ──
# new_rows = db.fetch_all("""
#     SELECT movie_id, MIN(target_date) AS first_date
#     FROM daily_box_office
#     WHERE rankOldAndNew = 'NEW'
#     GROUP BY movie_id
# """)
df_new = pd.DataFrame(new_rows)
df_new['first_date'] = pd.to_datetime(df_new['first_date'])
new_dates = df_new['first_date'].values.astype('datetime64[D]')

def count_same_week_releases(open_dt, new_dates, window=3):
    """개봉일 ±window일 이내의 다른 신작 수를 반환합니다 (자기 자신 제외)."""
    if pd.isna(open_dt):
        return 0
    open_np   = np.datetime64(open_dt, 'D')
    diff_days = np.abs((new_dates - open_np).astype(int))
    # diff==0 포함 후 -1 (자기 자신 제외)
    return max(0, int((diff_days <= window).sum()) - 1)

df['same_week_releases'] = df['open_date'].apply(
    lambda d: count_same_week_releases(d, new_dates)
)
print(f'✅ 경쟁 환경 피처(same_week_releases) 생성 완료')

# ── 6-2. 개봉 직전 7일 시장 평균 관객수 ──
# market_rows = db.fetch_all("""
#     SELECT target_date, total_audi
#     FROM daily_market_stats
#     ORDER BY target_date
# """)
df_market = pd.DataFrame(market_rows)
df_market['target_date'] = pd.to_datetime(df_market['target_date'])
df_market['total_audi']  = pd.to_numeric(df_market['total_audi'], errors='coerce')
df_market = df_market.set_index('target_date').sort_index()

def get_market_avg_7d(open_dt, df_market):
    """개봉일 직전 7일간의 시장 평균 관객수를 반환합니다."""
    if pd.isna(open_dt):
        return np.nan
    end_dt   = open_dt - pd.Timedelta(days=1)
    start_dt = open_dt - pd.Timedelta(days=7)
    window   = df_market.loc[start_dt:end_dt, 'total_audi']
    return float(window.mean()) if len(window) > 0 else np.nan

df['market_avg_audi_7d'] = df['open_date'].apply(
    lambda d: get_market_avg_7d(d, df_market)
)
# 결측값(시장 데이터 없음) → 전체 중앙값으로 대체
df['market_avg_audi_7d'] = df['market_avg_audi_7d'].fillna(df['market_avg_audi_7d'].median())
print(f'✅ 시장 활성도 피처(market_avg_audi_7d) 생성 완료')

✅ 경쟁 환경 피처(same_week_releases) 생성 완료
✅ 시장 활성도 피처(market_avg_audi_7d) 생성 완료


---
## 🗑️ 셀 7. 이상치 제거 및 최종 정리

### 제거 기준
| 조건 | 이유 |
|------|------|
| `total_audience <= 0` | 실질적인 상영 기록 없음 |
| `open_date` NaT | 개봉일 미상 → 시간/계절 피처 전부 무의미 |

In [20]:
before = len(df)

df = df[df['total_audience'] > 0]
df = df[df['open_date'].notna()]

after = len(df)
print(f'✅ 이상치 제거: {before}편 → {after}편 ({before - after}편 제거)')

# 최종 결측치 현황
null_summary = df.isna().sum()
null_summary = null_summary[null_summary > 0]
if len(null_summary) > 0:
    print(f'\n⚠️  잔여 결측치:')
    print(null_summary.to_string())
else:
    print('\n✅ 잔여 결측치 없음')

print(f'\n최종 데이터셋: {df.shape[0]}행 × {df.shape[1]}열')

✅ 이상치 제거: 3943편 → 3904편 (39편 제거)

✅ 잔여 결측치 없음

최종 데이터셋: 3904행 × 28열


---
## 💾 셀 8. CSV 저장 및 검증 (v1 완성)

> **이 셀까지가 v1 범위입니다. 이 아래에 새 피처 셀을 추가하고 VERSION을 올리세요.**

In [21]:
# ── 컬럼 순서 정의 ──
ID_COLS      = ['movie_id', 'title']
TARGET_COLS  = ['total_audience', 'log_audience', 'hit_class']

META_COLS    = ['runtime', 'rating_encoded', 'is_korean']
GENRE_COLS   = ['genre']
TIME_COLS    = ['open_month', 'open_day_of_week', 'is_summer', 'is_winter',
                'is_holiday_release', 'holiday_nearby_count']
STAR_COLS    = ['director_avg_audi', 'director_movie_count',
                'lead_actor_avg_audi', 'lead_actor_movie_count', 'cast_max_star_power']
BRAND_COLS   = ['distributor_avg_audi', 'distributor_movie_count',
                'producer_avg_audi', 'producer_movie_count']
MARKET_COLS  = ['same_week_releases', 'market_avg_audi_7d']

FEATURE_COLS = META_COLS + GENRE_COLS + TIME_COLS + STAR_COLS + BRAND_COLS + MARKET_COLS
FINAL_COLS   = ID_COLS + FEATURE_COLS + TARGET_COLS

# 존재하는 컬럼만 선택
final_cols = [c for c in FINAL_COLS if c in df.columns]
df_out     = df[final_cols].copy()

# ── 타입 명시 변환 (CSV 저장 시 문자열화 방지) ──
df_out['total_audience'] = df_out['total_audience'].astype(int)
df_out['hit_class']      = df_out['hit_class'].astype(int)

# CSV 저장
df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f'✅ {VERSION} 저장 완료: {OUTPUT_PATH}')
print(f'   shape: {df_out.shape[0]}편 × {df_out.shape[1]}컬럼')
print(f'\n컬럼 목록 ({len(final_cols)}개):')
for i, col in enumerate(final_cols, 1):
    print(f'  {i:2d}. {col}')

# 타겟 변수 최종 분포 확인
print(f'\n📊 타겟 변수 최종 분포:')
print(f'   total_audience  : 중앙값 {df_out["total_audience"].median():>12,.0f}명')
print(f'   log_audience    : 중앙값 {df_out["log_audience"].median():>12.3f}')
print(f'   hit_class 분포  :')
label_map = {0: '0 (~10만)', 1: '1 (10만~100만)', 2: '2 (100만~500만)', 3: '3 (500만~)'}
print(df_out['hit_class'].value_counts().sort_index().rename(label_map).to_string())

✅ v1 저장 완료: data/processed/feature_table_v1.csv
   shape: 3904편 × 26컬럼

컬럼 목록 (26개):
   1. movie_id
   2. title
   3. runtime
   4. rating_encoded
   5. is_korean
   6. genre
   7. open_month
   8. open_day_of_week
   9. is_summer
  10. is_winter
  11. is_holiday_release
  12. holiday_nearby_count
  13. director_avg_audi
  14. director_movie_count
  15. lead_actor_avg_audi
  16. lead_actor_movie_count
  17. cast_max_star_power
  18. distributor_avg_audi
  19. distributor_movie_count
  20. producer_avg_audi
  21. producer_movie_count
  22. same_week_releases
  23. market_avg_audi_7d
  24. total_audience
  25. log_audience
  26. hit_class

📊 타겟 변수 최종 분포:
   total_audience  : 중앙값       79,188명
   log_audience    : 중앙값       11.280
   hit_class 분포  :
hit_class
0 (~10만)         2092
1 (10만~100만)     1179
2 (100만~500만)     523
3 (500만~)         110


In [22]:
df['rating_encoded'].describe()

count    3904.000000
mean        1.345031
std         1.009086
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max         3.000000
Name: rating_encoded, dtype: float64

## Feature Table 전처리 전체 설명

---

### 전체 흐름

```
daily_box_office → 타겟 변수 추출
movies           → 영화 메타 피처
holidays         → 시간/계절 피처
people + movie_casting + daily_box_office → Star Power 피처
companys + company_part + daily_box_office → Brand Power 피처
daily_box_office + daily_market_stats → 경쟁 환경 피처
                    ↓
             이상치 제거
                    ↓
             CSV 저장
```

---

### 1. 타겟 변수

**왜 `MAX(audiAcc)`인가**

`audiAcc`는 일별 누적값입니다. 예를 들어 어떤 영화의 데이터가 이렇게 쌓입니다.

```
1일차: audiAcc = 100,000
2일차: audiAcc = 180,000
3일차: audiAcc = 210,000
```

`SUM`을 쓰면 490,000이 되어 완전히 틀린 값이 나옵니다. 가장 마지막 날의 누적값이 최종 총 관객수이므로 `MAX`를 씁니다.

---

**왜 `log_audience`를 실제 학습용 타겟으로 쓰는가**

관객수 분포를 보면 대부분의 영화는 수만 명 수준인데 일부 영화는 1,000만 명을 넘습니다. 이런 분포를 오른쪽으로 긴 꼬리(right-skewed)라고 합니다. 회귀 모델은 이런 분포에서 극단값에 지나치게 끌려가는 경향이 있어요.

`log1p`를 적용하면 분포가 정규분포에 가까워지고, 모델 입장에서 1,000만짜리 영화와 100만짜리 영화의 차이가 10배가 아니라 log 스케일로 완만하게 처리됩니다. 예측값을 다시 실제 관객수로 복원할 때는 `expm1(예측값)`을 쓰면 됩니다.

---

**왜 `hit_class`를 4구간으로 나눴는가**

회귀로 정확한 관객수를 맞추는 것보다 "이 영화가 흥행할까"라는 판단이 실용적으로 더 유용한 경우가 많습니다. 구간을 4개로 나눈 기준은 영화 산업에서 통용되는 규모 기준을 반영했습니다.

```
0: ~10만   → 소규모 (손익분기점 미달 가능성 높음)
1: 10만~100만 → 중규모
2: 100만~500만 → 흥행
3: 500만~  → 대흥행
```

---

### 2. 영화 메타 피처

**genre를 원핫인코딩 대신 문자열로 저장하는 이유**

원핫인코딩은 선형 모델(Ridge, Lasso)에 필요하지만 트리 계열 모델(XGBoost, LightGBM, RandomForest)은 문자열을 Label Encoding만 해도 충분히 처리합니다. 원핫으로 나누면 컬럼이 21개로 늘어나고 모델마다 전처리를 다르게 해야 하므로 원본 문자열로 저장하고 각 ML 노트북에서 필요한 방식으로 인코딩하는 구조로 잡았습니다.

**rating을 순서형으로 인코딩하는 이유**

관람등급은 단순한 범주가 아니라 순서가 있는 정보입니다. 전체관람가일수록 잠재 관객이 많고 청불일수록 적기 때문에 0~4의 순서형으로 인코딩하는 게 의미에 맞습니다.

---

### 3. 시간/계절 피처

**holiday_nearby_count를 ±7일로 잡은 이유**

영화 흥행에서 공휴일 효과는 당일만이 아니라 전후 1주일에 걸쳐 나타납니다. 설날이나 추석 연휴 직전에 개봉하면 연휴 내내 관객을 동원할 수 있기 때문에 당일 여부(is_holiday_release)와 별개로 근처 공휴일 수를 피처로 만들었습니다.

---

### 4. Star Power 피처

**핵심 원칙: 개봉일 이전 과거작만 참조**

만약 해당 영화 자신의 관객수를 Star Power 계산에 포함하면, 타겟 변수 정보가 피처에 새어 들어가는 데이터 누수(leakage)가 발생합니다. 그래서 `개봉일 이전에 개봉한 과거작`만 참조하도록 했습니다.

**신인 감독/배우는 0으로 설정하는 이유**

과거 데이터가 없다는 것 자체가 의미 있는 정보입니다. 0으로 설정하면 모델이 "검증된 실적 없음"으로 해석할 수 있고, 트리 계열 모델은 이 0값을 자연스럽게 분기 기준으로 활용합니다.

**cast_max_star_power를 추가한 이유**

평균 Star Power와 별개로, 출연진 중 단 한 명이라도 초대형 스타가 있으면 흥행에 큰 영향을 미칩니다. 최댓값을 별도 피처로 넣어 그 효과를 포착하려 했습니다.

---

### 5. Brand Power 피처

Star Power와 동일한 원칙으로 개봉일 이전 과거작만 참조합니다. 배급사와 제작사를 분리한 이유는 역할이 다르기 때문입니다. 배급사는 마케팅/배급망 역량을, 제작사는 제작 품질 역량을 반영합니다.

---

### 6. 경쟁 환경 피처

**same_week_releases**

같은 시기에 경쟁작이 많으면 스크린을 나눠 가져야 하므로 흥행에 불리합니다. `rankOldAndNew = 'NEW'`인 영화의 첫 등장일을 신작 개봉일로 보고, ±3일 이내 신작 수를 세었습니다.

**market_avg_audi_7d**

개봉 직전 7일의 시장 전체 관객수 평균입니다. 이 값이 높으면 극장을 찾는 사람이 많은 시기라는 뜻이므로 흥행에 유리한 환경 지표로 활용합니다.

---

### 7. 이상치 제거 기준

`total_audience <= 0`은 실질적인 상영 기록이 없는 데이터이고, `open_date`가 없으면 시간/계절 피처 전체가 의미 없어지므로 두 조건에 해당하는 행만 제거했습니다. 나머지는 최대한 보존했습니다.